# Clonal Homology Arm Generator

Generates clonal homology arm (HA) sequences and Golden Gate AMP primers from WT homology arm sequences, for cloning SGE libraries by Golden Gate.

## Input file format

A multi-sequence FASTA (`.fa`) where each target has **three entries**, named with the convention `{GENE}_{TARGET}_{TYPE}`:

| Sequence type | Name suffix | Description |
|---|---|---|
| AMP forward primer | `_AMP_F` | Forward amplification primer binding site (5' end of HA) |
| AMP reverse primer | `_AMP_R` | Reverse amplification primer binding site (3' end of HA), provided 5'→3' |
| Full homology arm | `_HA` or `_HA_recoded` | Full-length WT or recoded homology arm sequence |

Example:
```
>GENE1_X1A_AMP_F
ACGTACGTACGT
>GENE1_X1A_AMP_R
TTGGCCAATTGG
>GENE1_X1A_HA
ACGTACGTACGT...TTGGCCAATTGG
>GENE1_X1B_AMP_F
CGATCGATCGAT
>GENE1_X1B_AMP_R
AACCGGTTAACC
>GENE1_X1B_HA
CGATCGATCGAT...AACCGGTTAACC
```

## What the code does

1. **Reads** the input FASTA into a dataframe, fixing CR line endings if needed.
2. **Groups** sequences by gene/target (`{GENE}_{TARGET}`).
3. **Pre-checks** all input sequences for REase sites. Any hits are flagged with a warning and saved to a separate FASTA for review — sequences with pre-existing REase sites will require recoding before proceeding.
4. **Trims** the full HA to the AMP primer binding sites (with 8 bp overlap on each end) to produce 5' and 3' homology arm fragments.
5. **Builds the clonal HA** by joining: `5' HA + stuffer sequence + 3' HA`. The stuffer contains PaqCI (CACCTGC) sites at both ends for Golden Gate cloning.
6. **Builds Golden Gate AMP primers** by appending standard GG stuffer overhangs to each AMP primer.
7. **Validates** outputs by checking that each clonal HA contains exactly 2 REase sites and each primer contains exactly 1.
8. **Saves** the clonal HAs and GG AMP primers as separate FASTA files.

**Note that hardcoded stuffers and automated REase checking are for PaqCI sites. Adjust if using different REase**

In [ ]:
import pandas as pd
from pyfaidx import Fasta
from Bio.Seq import Seq
import warnings
import datetime

In [ ]:
input_fa = '.fa' #Input .fa containing WT homology arm sequences
stuffer_seq = 'GCAGGTGTTTCTACATGTATTTCCTTCTGGTCTTTTCCCACCATCATATACATCTTACAAGCTTGCAATCTTAGTGTGCTTAAACACCTGC' #Standard stuffer with REase sites at both ends
save_outputs = True #Save outputs?

date = datetime.datetime.now().strftime('%Y%m%d') #Gets date for file name

test_output = f'{date}.fa' #Outuput path for input sequences with pre-existing REase sites

#Sets output paths
if save_outputs:
    clonal_ha_output = f'{date}.fa'
    ggamp_output = f'{date}.fa'


gg_amp_stuffers = ['ATGTCACCTGCCACT', 'CTAGCACCTGCCACA'] #Standard stuffers to build golden gate AMP primers


#REase site that will be checked against
rease_site = 'CACCTGC' #Currently PaqCI
rev_comp_rease_site = str(Seq(rease_site).reverse_complement()) #Checks rev. complement as well

In [ ]:
def fix_line_endings(fasta_path): #Fixes any encoding issues with .fa
    """Convert CR-only line endings to LF in place."""
    with open(fasta_path, 'rb') as f:
        data = f.read()
    if b'\r\n' not in data and b'\r' in data:
        with open(fasta_path, 'wb') as f:
            f.write(data.replace(b'\r', b'\n'))

def read_fasta_to_df(fasta_path): #Reads .fa to dataframe
    seq_names = []
    seqs = []

    fix_line_endings(fasta_path)
    fa = Fasta(fasta_path)
    
    for key in fa.keys():
        seq_names.append(key)
        seq = str(fa[key]).upper()

        seqs.append(seq)
    

    df = pd.DataFrame({'seq_name': seq_names,
                       'seqs': seqs})
    
    return df

In [ ]:
df = read_fasta_to_df(input_fa)

df['gene_target'] = df['seq_name'].transform(lambda x: x.split('_')[0] + '_' + x.split('_')[1]) #Adds column in format of {GENE_NAME}_{target} for grouping

missing_values = []
for gene_target, group in df.groupby('gene_target'):
    if len(group) < 3:
        missing_values.append(gene_target)

if len(missing_values) != 0:
    print(f'{missing_values} will be removed prior to proceeding...')
    df = df[~df['gene_target'].isin(missing_values)]
    warnings.warn(f'Missing sequences for:{missing_values}')
elif len(missing_values)==0:
    print('No missing values detected!')
    

In [ ]:
def check_seqs(df = df, output_path = test_output, save = True): #Pre-clonal HA generation check for extra REase sites (expect no sequences to have REase sites)
    hit_seq_names = []
    hit_seqs = []

    for index, row in df.iterrows():
        name = row['seq_name']
        seq = row['seqs']
        
        if rease_site in seq or rev_comp_rease_site in seq:
            hit_seq_names.append(name)
            hit_seqs.append(seq)
    
    if len(hit_seq_names) != 0:
        for name in hit_seq_names:
            print(f'Detected REase site in {name}')

        if save:
            print('Writing .fa with REase site hits...')

            i = 0
            with open(output_path, 'w') as out:
                while i < len(hit_seq_names):
                    seq_name = hit_seq_names[i]
                    seq = hit_seqs[i]

                    out.write(f'>{seq_name}\n')
                    out.write(f'{seq}\n')

                    i += 1
                        
            print(f'File written to: {output_path}')

        raise ValueError(f'HALTING: REase sites detected in {len(hit_seq_names)} input sequence(s). Recode before proceeding.')

    elif len(hit_seq_names) == 0:
        print('No extraneous REase sites detected!')

check_seqs()

In [ ]:
clonal_homology_arms = {} #Dictionary to store clonal HA sequences

gg_amp_primers = {} #Dictionary to store golden gate AMP primers

for gene_target, group in df.groupby('gene_target'): #Groups by gene_target column
    print(gene_target)

    amp_f = group.loc[group['seq_name'].str.contains('AMP_F')]['seqs'].tolist()[0] #Gets AMP_F
    amp_r = str(Seq(group.loc[group['seq_name'].str.contains('AMP_R')]['seqs'].tolist()[0]).reverse_complement()) #Gets reverse complement of AMP_R
    full_ha = group.loc[group['seq_name'].str.contains('_HA')]['seqs'].tolist()[0] #Gets full homology arm


    amp_f_index_start = full_ha.find(amp_f) #Finds index at which AMP_F binds #(5' end)
    amp_r_index_start = full_ha.find(amp_r) #Finds index at which AMP_R binds (3' end)

    length_amp_r = len(amp_r) #Gets length of AMP_R (needed for clonal HA)

    five_prime_ha = full_ha[0:amp_f_index_start + 8] #Adds 8 bp of AMP_F primer to 5' HA
    three_prime_ha = full_ha[(length_amp_r + amp_r_index_start) -8:] #Adds 8 bp of AMP_R primer to 3' HA

    #Prints HAs
    print(f'This is the 5 prime homology arm: {five_prime_ha}')
    print(f'This is the 3 prime homology arm: {three_prime_ha}')

    clonal_ha = five_prime_ha + stuffer_seq + three_prime_ha #Builds full HA

    clonal_homology_arms[f'{gene_target}_clonal_HA'] = clonal_ha #Full HA added to dictionary


    amp_r_five_to_three = group.loc[group['seq_name'].str.contains('AMP_R')]['seqs'].tolist()[0] #Gets AMP_R in proper orientation
    gg_amp_f = gg_amp_stuffers[0] + amp_f #Builds golden gate AMP_F
    gg_amp_r = gg_amp_stuffers[1] + amp_r_five_to_three #Builds golden gate AMP_R

    #Primers added to dict
    gg_amp_primers[f'{gene_target}_ggAMP_F'] = gg_amp_f
    gg_amp_primers[f'{gene_target}_ggAMP_R'] = gg_amp_r

print(gg_amp_primers) #Primers printed

In [ ]:
def check_outputs(clonal_output = clonal_homology_arms, amp_output = gg_amp_primers): #Helper function to ensure outputs have the proper number of REase sites (2 for HAs, 1 for primers)
    dict_tuples = [(clonal_output, 2), (amp_output, 1)]

    for dict, max_rease in dict_tuples:

        for seq_name, seq in dict.items():
            fwd_count = seq.count(rease_site)
            rev_count = seq.count(rev_comp_rease_site)

            total_count = fwd_count + rev_count

            if total_count > max_rease:
                warnings.warn(f'More than {max_rease} REase sites detected for {seq_name}. Check design before proceeding: {seq}')
            elif total_count < max_rease:
                warnings.warn(f'Fewer than {max_rease} REase sites detected for {seq_name}. Check design before proceeding: {seq}')
    
    print('All outputs checked. If no warnings appeared, no extra REase sites were detected')
check_outputs()


In [ ]:
#Saves outputs
if save_outputs:
    with open(clonal_ha_output, 'w')as out:
        for clone in clonal_homology_arms.keys():
            seq = clonal_homology_arms[clone]

            out.write(f'>{clone}\n')
            out.write(f'{seq}\n')
    
    with open(ggamp_output, 'w')as out:
        for primer in gg_amp_primers.keys():
            seq = gg_amp_primers[primer]

            out.write(f'>{primer}\n')
            out.write(f'{seq}\n')